# Lab 6 – Extended Analysis
**New Companies:** Amazon (`AMZN`), Tesla (`TSLA`), NVIDIA (`NVDA`)

**New Analyses (not in the original lab):**
1. Bollinger Bands
2. Relative Strength Index (RSI)
3. Value at Risk (VaR)


## 0. Setup & Data Download

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ── New companies (different from AAPL / GOOG / MSFT in the original lab) ──
tickers = ['AMZN', 'TSLA', 'NVDA']

raw = yf.download(tickers, start='2020-01-01', end='2025-01-01')
data = raw['Close']

print(data.tail())

---
## Analysis 1 – Bollinger Bands

Bollinger Bands show a stock's price volatility by plotting an upper and lower band
two standard deviations away from a rolling mean.  
- **Price above upper band** → possibly overbought  
- **Price below lower band** → possibly oversold

In [ ]:
def compute_bollinger(series: pd.Series, window: int = 20, num_std: float = 2.0):
    """Return DataFrame with price, rolling mean (SMA), upper and lower Bollinger Bands."""
    df = pd.DataFrame({'Close': series})
    df['SMA'] = df['Close'].rolling(window).mean()
    df['STD'] = df['Close'].rolling(window).std()
    df['Upper'] = df['SMA'] + num_std * df['STD']
    df['Lower'] = df['SMA'] - num_std * df['STD']
    df['%B'] = (df['Close'] - df['Lower']) / (df['Upper'] - df['Lower'])  # 0–1 position
    return df.dropna()

fig, axes = plt.subplots(len(tickers), 1, figsize=(12, 5 * len(tickers)), sharex=False)

for ax, ticker in zip(axes, tickers):
    bb = compute_bollinger(data[ticker])

    ax.fill_between(bb.index, bb['Upper'], bb['Lower'], alpha=0.15, color='blue', label='BB Band')
    ax.plot(bb.index, bb['Close'], label=f'{ticker} Price', linewidth=1.2)
    ax.plot(bb.index, bb['SMA'],   label='20-Day SMA', linestyle='--', linewidth=1)
    ax.plot(bb.index, bb['Upper'], label='Upper Band', linestyle=':', color='red', linewidth=0.9)
    ax.plot(bb.index, bb['Lower'], label='Lower Band', linestyle=':', color='green', linewidth=0.9)

    ax.set_title(f'{ticker} – Bollinger Bands (20-day, ±2σ)')
    ax.set_ylabel('Price (USD)')
    ax.legend(loc='upper left', fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.tight_layout()
plt.show()

# Print the most recent %B values to gauge current position
print('\nMost recent %B value (0 = at lower band, 1 = at upper band):')
for ticker in tickers:
    bb = compute_bollinger(data[ticker])
    print(f'  {ticker}: {bb["%B"].iloc[-1]:.3f}')

---
## Analysis 2 – Relative Strength Index (RSI)

RSI is a momentum oscillator that measures the speed and magnitude of price changes on a 0–100 scale.  
- RSI **> 70** → overbought signal  
- RSI **< 30** → oversold signal  

The standard look-back period is **14 trading days**.

In [ ]:
def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    """Wilder's smoothed RSI."""
    delta = series.diff()
    gain  = delta.clip(lower=0)
    loss  = (-delta).clip(lower=0)

    # First average
    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()

    # Wilder smoothing for subsequent rows
    for i in range(period, len(series)):
        avg_gain.iloc[i] = (avg_gain.iloc[i - 1] * (period - 1) + gain.iloc[i]) / period
        avg_loss.iloc[i] = (avg_loss.iloc[i - 1] * (period - 1) + loss.iloc[i]) / period

    rs  = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi

fig, axes = plt.subplots(len(tickers) * 2, 1, figsize=(12, 4 * len(tickers) * 2),
                         gridspec_kw={'height_ratios': [2, 1] * len(tickers)})

for i, ticker in enumerate(tickers):
    price_ax = axes[i * 2]
    rsi_ax   = axes[i * 2 + 1]

    rsi_series = compute_rsi(data[ticker])

    # Price panel
    price_ax.plot(data.index, data[ticker], label=f'{ticker} Price')
    price_ax.set_title(f'{ticker} – Price & RSI(14)')
    price_ax.set_ylabel('Price (USD)')
    price_ax.legend(fontsize=8)

    # RSI panel
    rsi_ax.plot(rsi_series.index, rsi_series, color='purple', linewidth=1)
    rsi_ax.axhline(70, color='red',   linestyle='--', linewidth=0.9, label='Overbought (70)')
    rsi_ax.axhline(30, color='green', linestyle='--', linewidth=0.9, label='Oversold (30)')
    rsi_ax.fill_between(rsi_series.index, 70, rsi_series, where=(rsi_series > 70),
                        alpha=0.25, color='red')
    rsi_ax.fill_between(rsi_series.index, 30, rsi_series, where=(rsi_series < 30),
                        alpha=0.25, color='green')
    rsi_ax.set_ylim(0, 100)
    rsi_ax.set_ylabel('RSI')
    rsi_ax.legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

print('\nLatest RSI values:')
for ticker in tickers:
    rsi_val = compute_rsi(data[ticker]).dropna().iloc[-1]
    signal  = 'Overbought' if rsi_val > 70 else ('Oversold' if rsi_val < 30 else 'Neutral')
    print(f'  {ticker}: RSI = {rsi_val:.1f}  →  {signal}')

---
## Analysis 3 – Value at Risk (VaR)

VaR estimates the **maximum expected loss** over a given time horizon at a chosen confidence level.

We compute two flavours:
- **Historical VaR** – uses the empirical distribution of daily returns  
- **Parametric (Gaussian) VaR** – assumes returns are normally distributed

A 95 % VaR of −3 % means: on a typical day, losses should not exceed 3 % with 95 % confidence.

In [ ]:
from scipy.stats import norm

daily_returns = data.pct_change().dropna()

confidence_levels = [0.90, 0.95, 0.99]

# ── Compute VaR ──────────────────────────────────────────────────────────────
results = []
for ticker in tickers:
    r = daily_returns[ticker]
    for cl in confidence_levels:
        hist_var  = np.percentile(r, (1 - cl) * 100)          # Historical
        param_var = norm.ppf(1 - cl, loc=r.mean(), scale=r.std())  # Parametric
        results.append({
            'Ticker': ticker,
            'Confidence': f'{int(cl*100)}%',
            'Historical VaR': f'{hist_var:.4%}',
            'Parametric VaR': f'{param_var:.4%}'
        })

var_df = pd.DataFrame(results)
print(var_df.to_string(index=False))

# ── Plot return distributions with VaR cut-offs ───────────────────────────────
fig, axes = plt.subplots(1, len(tickers), figsize=(14, 4), sharey=False)

for ax, ticker in zip(axes, tickers):
    r = daily_returns[ticker]
    ax.hist(r, bins=80, color='steelblue', edgecolor='white', alpha=0.8, density=True)

    for cl, color in zip(confidence_levels, ['gold', 'orange', 'red']):
        hist_var = np.percentile(r, (1 - cl) * 100)
        ax.axvline(hist_var, color=color, linestyle='--', linewidth=1.5,
                   label=f'VaR {int(cl*100)}% = {hist_var:.2%}')

    # Overlay normal curve
    x = np.linspace(r.min(), r.max(), 300)
    ax.plot(x, norm.pdf(x, r.mean(), r.std()), 'k-', linewidth=1.2, label='Normal fit')

    ax.set_title(f'{ticker} – Daily Return Distribution & VaR')
    ax.set_xlabel('Daily Return')
    ax.set_ylabel('Density')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

---
## Summary

| Section | Companies | Method |
|---------|-----------|--------|
| Analysis 1 | AMZN, TSLA, NVDA | **Bollinger Bands** (20-day, ±2σ) |
| Analysis 2 | AMZN, TSLA, NVDA | **RSI** (14-day Wilder smoothing) |
| Analysis 3 | AMZN, TSLA, NVDA | **Value at Risk** (Historical & Parametric, 90/95/99%) |

All three analyses are **not present** in the original Lab 6.